# Classic SPL weekly gate

Runs before weekly scoring. Decides whether this week's score run should go ahead.

1. Computes `end_date`: the most recent Saturday before today (US/Central), same rule as the Domino weekly job (`run_weekly_spl.py`, `end_date_day_of_week: 5`). A manual `end_date_override` wins.
2. Skips if `public_results` already has this `end_date` (week already scored).
3. Checks that `ltv_base` has data through `end_date` (upstream refreshed). The eltv inputs must be copied in before this notebook runs.
4. Exits with a JSON decision that the pipeline reads with
   `@json(activity('Weekly gate').output.result.exitValue)`.


In [ ]:
# Parameters (the pipeline overrides these)
end_date_override = ""       # "YYYYMMDD" to force a specific end date
today_override = ""          # "YYYYMMDD" to test the date logic as if today were that day
git_checkout = "NB_Classic_SPL_v10.0.0"
model_version = "10.0.0"
DATE_COL = "release_day_pol"  # date column in ltv_base used for the upstream check; confirm it


In [ ]:
import json
import datetime
from zoneinfo import ZoneInfo

import notebookutils
from pyspark.sql import functions as F
from pyspark.sql import types as T

import ltv_helpers.fabric_pipeline_helpers as fh
from classic_spl_ltv.config import run_paths

TZ = ZoneInfo("America/Chicago")
END_DATE_WEEKDAY = 5  # Saturday, Python weekday (Mon=0)


def last_saturday(today: datetime.date) -> datetime.date:
    """Most recent Saturday strictly before today. Same formula as run_weekly_spl.py line 348."""
    return today + datetime.timedelta(days=(END_DATE_WEEKDAY - today.weekday()) % 7 - 7)


## Self-check of the date rule

In [ ]:
cases = {
    "2026-11-04": "2026-10-31",  # Wednesday -> previous Saturday
    "2026-11-06": "2026-10-31",  # Friday
    "2026-11-07": "2026-10-31",  # Saturday -> the Saturday before, not today
    "2026-11-08": "2026-11-07",  # Sunday -> yesterday
}
for d, expected in cases.items():
    got = last_saturday(datetime.date.fromisoformat(d))
    assert got == datetime.date.fromisoformat(expected), f"{d}: got {got}, expected {expected}"
print("date rule OK")


## 1. End date

In [ ]:
if today_override:
    today = datetime.datetime.strptime(today_override, "%Y%m%d").date()
else:
    today = datetime.datetime.now(TZ).date()

forced = bool(end_date_override)
if forced:
    end_date = datetime.datetime.strptime(end_date_override, "%Y%m%d").date()
else:
    end_date = last_saturday(today)

print(f"today={today} ({today:%A})  end_date={end_date}  mode={'override' if forced else 'auto'}")


## 2. Already scored this week?

In [ ]:
p = run_paths.make_paths("score", git_checkout, model_version, end_date=None)

last_published = None
if notebookutils.fs.exists(p.public_results):
    last_published = fh.read_delta_table(spark, p.public_results).agg(F.max("end_date")).first()[0]

already_done = last_published is not None and last_published >= end_date
print(f"public_results: {p.public_results}")
print(f"last published end_date={last_published}  already_done={already_done}")


## 3. Upstream ready?

In [ ]:
def read_any(path):
    if notebookutils.fs.exists(f"{path}/_delta_log"):
        return spark.read.format("delta").load(path)
    return spark.read.parquet(path)


def as_date(df, col):
    """Normalize the date column whether it is a date, timestamp, or YYYYMMDD / YYYY-MM-DD string or int."""
    dtype = df.schema[col].dataType
    if isinstance(dtype, T.DateType):
        return F.col(col)
    if isinstance(dtype, T.TimestampType):
        return F.to_date(F.col(col))
    s = F.col(col).cast("string")
    return F.coalesce(F.to_date(s, "yyyyMMdd"), F.to_date(s, "yyyy-MM-dd"))


assert notebookutils.fs.exists(p.ltv_base), f"ltv_base not found: {p.ltv_base} (run the eltv import first)"
base = read_any(p.ltv_base)

if DATE_COL not in base.columns:
    candidates = [c for c in base.columns if any(k in c.lower() for k in ("date", "day", "dt"))]
    raise KeyError(f"{DATE_COL} not in ltv_base. Date-like columns: {candidates}")

max_date = base.agg(F.max(as_date(base, DATE_COL))).first()[0]
upstream_ready = max_date is not None and max_date >= end_date
print(f"ltv_base: {p.ltv_base}")
print(f"ltv_base max {DATE_COL}={max_date}  upstream_ready={upstream_ready}")


## 4. Decision

In [ ]:
run = forced or (not already_done and upstream_ready)

if forced:
    reason = "manual end_date override"
elif already_done:
    reason = "week already scored"
elif not upstream_ready:
    reason = "upstream ltv_base not refreshed through end_date"
else:
    reason = "ready"

result = {
    "run": run,
    "end_date": f"{end_date:%Y%m%d}",
    "reason": reason,
    "already_done": already_done,
    "upstream_ready": upstream_ready,
    "is_last_day": today.weekday() == 4,  # Friday: last scheduled try of the week
    "ltv_base_max_date": str(max_date),
    "last_published_end_date": str(last_published),
}
print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))
